In [ ]:
# import packages
import pandas as pd
import numpy as np
import os, dotenv, geopy, folium

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans
from sklearn import linear_model

from foursquare_api import fetch_foursquare_raw, fetch_foursquare_venues

# Datasets Used

In [ ]:
# read the datasets
original_rating = pd.read_csv('location_rating.csv')
original_rating.head()

# Geological Data Retrieval

In [ ]:
# get the latitude and longitude of the locations
geolocator = geopy.geocoders.Nominatim(user_agent = "foursquare_agent")

latitude_list = []
longitude_list = []

for address in original_rating['location']:
    try:
        location = geolocator.geocode(address, timeout = 10)
        if location:
            latitude_list.append(location.latitude)
            longitude_list.append(location.longitude)
        else:
            latitude_list.append(None)
            longitude_list.append(None)
    except geopy.exc.GeocoderTimedOut:
        latitude_list.append(None)
        longitude_list.append(None)
        print(f"Timeout: {address}")

original_rating['latitude'] = latitude_list
original_rating['longitude'] = longitude_list

In [ ]:
# create map of Toronto using latitude and longitude values
map_original = folium.Map(location=[original_rating['latitude'].mean(), original_rating['longitude'].mean()], zoom_start = 4)

# add markers to map
for lat, lng, label, rating in zip(original_rating['latitude'], original_rating['longitude'], original_rating['location'], original_rating['rating']):
    folium.CircleMarker([lat, lng], 
                        radius = 5,
                        tooltip = f"{label}: {rating}",
                        color = 'blue', fill = True, fill_color = '#3186cc', fill_opacity = 0.7).add_to(map_original)

#show the map
map_original

# Foursquare Data Retrieval

In [ ]:
# import the api_key for Foursquare
dotenv.load_dotenv("../personal_envs/neighborhood-preference-prediction.env", override=True)
api_key = os.getenv("toronto_venue_clustering")

In [ ]:
# fetch the foursquare venue data
original_rating_venue = fetch_foursquare_venues(api_key, original_rating, 'latitude', 'longitude', 1000, 50)
original_rating_venue = original_rating_venue[original_rating_venue['venue_category'].notna()]

In [ ]:
# get the dummy data of category
mlb = MultiLabelBinarizer()
category_dummies = pd.DataFrame(mlb.fit_transform(original_rating_venue['venue_category']), columns = mlb.classes_, index = original_rating_venue.index)
print(f'There are {len(category_dummies.columns)} uniques categories.')

rating_venues_dummy = pd.concat([original_rating_venue[['location']], category_dummies], axis=1)

In [ ]:
# get the dummy data grouped
rating_venues_dummy_group = rating_venues_dummy.groupby('location').mean().reset_index()
rating_venues_dummy_group = rating_venues_dummy_group.merge(original_rating[['location', 'rating']], on = 'location', how = 'inner')

rating_venues_dummy_group.head(3)

# Model Building

In [ ]:
# train the model
lr = linear_model.LinearRegression()
x = np.asanyarray(rating_venues_dummy_group[rating_venues_dummy_group.columns[1:-1]])
y = np.asanyarray(rating_venues_dummy_group[rating_venues_dummy_group.columns[-1]])
lr.fit (x, y)

# Testing Data Prediciton

In [ ]:
# get the location for prediction
place_test = 'Athens, Greece'
test_data = pd.DataFrame([[place_test]] , columns = ['location']) 

# get the latitude and longitude of the location for prediction
geolocator = geopy.geocoders.Nominatim(user_agent="foursquare_agent")

latitude_list = []
longitude_list = []

for address in test_data['location']:
    try:
        location = geolocator.geocode(address, timeout = 10)
        if location:
            latitude_list.append(location.latitude)
            longitude_list.append(location.longitude)
        else:
            latitude_list.append(None)
            longitude_list.append(None)
    except geopy.exc.GeocoderTimedOut:
        latitude_list.append(None)
        longitude_list.append(None)
        print(f"Timeout: {address}")

test_data['latitude'] = latitude_list
test_data['longitude'] = longitude_list

In [ ]:
# fetch the foursquare venue data
test_df = fetch_foursquare_venues(api_key, test_data, 'latitude', 'longitude', 1000, 50)

In [ ]:
# get the dummy data of category
mlb = MultiLabelBinarizer()
test_category_dummies = pd.DataFrame(mlb.fit_transform(test_df['venue_category']), columns = mlb.classes_, index = test_df.index)
print(f'There are {len(test_category_dummies.columns)} uniques categories.')

test_rating_venues_dummy = pd.concat([test_df[['location']], category_dummies], axis=1)

In [ ]:
# get the dummy data grouped
test_rating_venues_dummy_group = test_rating_venues_dummy.groupby('location').mean().reset_index()
test_rating_venues_dummy_group.head(3)

In [ ]:
# match the test columns to original
test_rating_venues_dummy_group = test_rating_venues_dummy_group.reindex(
    columns=rating_venues_dummy_group.columns,
    fill_value=0
)

In [ ]:
# The prediction of the test place
test_prediction= lr.predict(np.asanyarray(test_rating_venues_dummy_group[rating_venues_dummy_group.columns[1:-1]]))
print(f'The predicted rate of {place_test} will be {round(test_prediction[0], 2)}.')